# 2.3 bis — Méthode par dictionnaire et règles, DIC-v3

**Point de départ : DIC-v2** (notebook 2.3), reconstruite ici à l'identique : même dictionnaire, même configuration
(casse insensible, seuil 0,3, regex), mêmes cinq règles. Le notebook vérifie qu'on retrouve ses prédictions.

**Ce qui est ajouté** (section 2.3 du livret : « choix de casse, de variantes et de priorité ») :

| # | Ajout | Erreur visée sur VALIDATION (DIC-v2) | Appris sur |
|---|---|---|---|
| A | **Filtre corrigé** : une occurrence incluse dans une mention plus longue ne compte plus comme « non annotée » | 106 mentions manquées dont la forme est retirée à tort (*cancer, edema…*) | TRAIN |
| B | **Variantes singulier / pluriel** d'une entrée, si la variante n'apparaît jamais dans TRAIN | 54 mentions manquées (*tumors, anemias…*) | TRAIN |
| C | **Extension vers la droite** : suites de 1 à 3 mots qui prolongent une mention dans TRAIN (≥ 3 fois, ≥ 70 %) | 514 frontières trop courtes (*migraine* → *migraine with aura*) | TRAIN |
| D | **Propagation dans le document** : une forme reconnue une fois est reconnue à ses autres occurrences (même casse) | 88 mentions manquées trouvées ailleurs dans le même résumé | — |

Les seuils de C reprennent ceux de la règle des modificateurs à gauche (DIC-v2), sans nouveau réglage. Chaque ajout
est mesuré seul puis avec les autres ; il est gardé s'il améliore le F1 VALIDATION. TEST ne sert à rien construire.

In [1]:
import sys, json, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import Mention, erreurs
from tals.spans import tokeniser_avec_offsets
from src.cdr import CATEGORIES, charger_jsonl, mentions_reference, decouper_phrases
from src.dictionnaire import DictionnaireCDR, ConfigDictionnaire, REGEX
from src.extraction import tableau_scores, ecrire_predictions, lire_predictions

RESULTATS = RACINE / "resultats" / "extraction"
SEED = 42

documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")  # écrit par 2.2
train = [d for d in documents if d.partition == "train"]
validation = [d for d in documents if d.partition == "validation"]
test = [d for d in documents if d.partition == "test"]
ref_val = mentions_reference(validation)
textes_val = {d.document_id: d.texte for d in validation}
print(len(train), len(validation), len(test), "documents ;", len(ref_val), "mentions de référence en VALIDATION")

500 500 500 documents ; 9591 mentions de référence en VALIDATION


## DIC-v2 reconstruite

Configuration retenue au 2.3 (casse insensible, seuil 0,3, regex).

In [2]:
CONFIG = ConfigDictionnaire("insensible", 0.3, True)
dic_v2 = DictionnaireCDR(CONFIG).construire(train)
DICO_COURANT = dic_v2

In [3]:
# Règles 1 et 2 : apprises sur TRAIN uniquement
compte = defaultdict(Counter)
for d in train:
    types = {(m.debut, m.fin): m.type for m in d.mentions}
    for r in re.finditer(r"\b[a-z]{7,}\b", d.texte):
        for n in (4, 5, 6):
            compte[r.group()[-n:]][types.get(r.span(), "O")] += 1
SUFFIXES = {}
for suffixe, c in compte.items():
    type_, k = max(((t, c[t]) for t in CATEGORIES), key=lambda x: x[1])
    if sum(c.values()) >= 8 and k / sum(c.values()) >= 0.8:
        SUFFIXES[suffixe] = type_
# On garde le suffixe le plus court quand un plus long donne le même type (« amine » couvre « tamine »)
SUFFIXES = {s: t for s, t in SUFFIXES.items() if not any(o != s and s.endswith(o) and SUFFIXES[o] == t for o in SUFFIXES)}
REGEX_SUFFIXES = {t: re.compile(r"\b[A-Za-z][a-z]{3,}(?:" + "|".join(sorted((s for s, u in SUFFIXES.items() if u == t), key=len, reverse=True)) + r")\b")
                  for t in CATEGORIES}


def mot_avant(texte, pos):
    """Mot qui précède immédiatement la position `pos` : (mot en minuscules, début) ou None."""
    debut = max(0, pos - 40)
    r = re.search(r"([A-Za-z][\w']*)\s+$", texte[debut:pos])
    return (r.group(1).lower(), debut + r.start(1)) if r else None


inclus, exclus = Counter(), Counter()
for d in train:
    maladies = {(m.debut, m.fin) for m in d.mentions if m.type == "Disease"}
    for _, a, b, type_ in dic_v2._correspondances(d.texte):
        mot = mot_avant(d.texte, a) if type_ == "Disease" else None
        if mot is None:
            continue
        if any(x <= mot[1] and y >= b for x, y in maladies):
            inclus[mot[0]] += 1
        elif (a, b) in maladies:
            exclus[mot[0]] += 1
MODIFICATEURS = {w for w in inclus if inclus[w] >= 3 and inclus[w] / (inclus[w] + exclus[w]) >= 0.7}

for t in CATEGORIES:
    print(f"Suffixes {t} ({sum(u == t for u in SUFFIXES.values())}) :", sorted(s for s, u in SUFFIXES.items() if u == t))
print(f"\nModificateurs à gauche ({len(MODIFICATEURS)}) :", sorted(MODIFICATEURS))

Suffixes Chemical (81) : ['acin', 'aine', 'aline', 'amate', 'amil', 'amine', 'amol', 'amycin', 'anol', 'anyl', 'aride', 'arin', 'atin', 'axel', 'bine', 'chol', 'cium', 'dine', 'done', 'eine', 'enol', 'epam', 'erin', 'erol', 'fene', 'hine', 'hium', 'holine', 'honium', 'icin', 'idol', 'iine', 'iline', 'imus', 'inol', 'irin', 'kinin', 'lcet', 'llin', 'lline', 'lone', 'mide', 'mole', 'ndol', 'nidate', 'ntin', 'oban', 'ocin', 'odopa', 'ofol', 'ohol', 'olam', 'olide', 'olol', 'osin', 'othane', 'otide', 'phen', 'pine', 'pion', 'pril', 'ptan', 'rine', 'rone', 'rtan', 'sine', 'sium', 'sone', 'tasy', 'tine', 'tinine', 'toin', 'tone', 'trogen', 'uine', 'urane', 'xate', 'xine', 'xone', 'zine', 'zole']
Suffixes Disease (27) : ['ache', 'alepsy', 'alic', 'bosis', 'brosis', 'cities', 'enia', 'haly', 'hmia', 'hosis', 'hrosis', 'iasis', 'isia', 'lemia', 'lsions', 'mnesia', 'nemia', 'raine', 'rmia', 'sias', 'toma', 'uising', 'uria', 'xicity', 'xiety', 'zure', 'zures']

Modificateurs à gauche (49) : ['abd

In [4]:
EXCLUS_INDUIT = {"drug", "drugs", "stress", "therapy", "anesthesia", "handling", "blockade", "stimulus", "contrast",
                 "acid", "exercise", "surgery", "treatment", "injury", "ischemia", "hypoxia", "radiation", "diet",
                 "agonist", "antagonist", "neuroleptic", "analgesic", "psychostimulant", "the", "both", "rapid",
                 "maximal", "single", "combined", "chronic", "saline"}
ADJECTIF = re.compile(r".*(?:al|ic|ive|ous|ed|ing|ar)")


def libre(m, P):
    """Vrai si la mention ne chevauche aucune mention de P (P : mentions d'un même document)."""
    return all(m.fin <= x.debut or x.fin <= m.debut for x in P)


def r_suffixes(d, P):
    P = set(P)
    for type_, motif in REGEX_SUFFIXES.items():
        for r in motif.finditer(d.texte):
            m = Mention(d.document_id, r.start(), r.end(), type_)
            if libre(m, P):
                P.add(m)
    return P


def r_modificateurs(d, P):
    P = set(P)
    for m in sorted((m for m in P if m.type == "Disease"), key=lambda m: -m.debut):
        mot = mot_avant(d.texte, m.debut)
        if mot and mot[0] in MODIFICATEURS:
            etendue = Mention(m.doc, mot[1], m.fin, "Disease")
            if libre(etendue, P - {m}):
                P = (P - {m}) | {etendue}
    return P


def r_induit(d, P):
    P = set(P)
    for motif in (r"(?<![\w-])([A-Za-z][A-Za-z0-9]{3,})-induced\b", r"\binduced by (?:the |a )?([A-Za-z][A-Za-z0-9-]{3,})\b"):
        for r in re.finditer(motif, d.texte):
            mot = r.group(1).lower()
            if mot in EXCLUS_INDUIT or ADJECTIF.fullmatch(mot):
                continue
            m = Mention(d.document_id, *r.span(1), "Chemical")
            if libre(m, P):
                P.add(m)
    return P


def r_sigles(d, P):
    t, P = d.texte, set(P)
    for r in re.finditer(r"\s\(([A-Za-z0-9][A-Za-z0-9-]{0,9})\)", t):
        sigle = r.group(1)
        if not re.search(r"[A-Z]", sigle):
            continue
        longues = [m for m in P if m.fin == r.start()]
        occurrences = [(x.start(), x.end()) for x in re.finditer(r"(?<![\w-])" + re.escape(sigle) + r"(?![\w-])", t)]
        if longues:
            type_ = longues[0].type
            P = {m for m in P if not ((m.debut, m.fin) in occurrences and m.type != type_)}
            for a, b in occurrences:
                m = Mention(d.document_id, a, b, type_)
                if libre(m, P):
                    P.add(m)
        elif DICO_COURANT._cle(sigle) not in DICO_COURANT.entrees:
            P = {m for m in P if (m.debut, m.fin) not in occurrences}
    return P


def r_anti(d, P):
    return {m for m in P if not d.texte[m.debut:m.fin].lower().startswith("anti")}


REGLES = {"suffixes appris": r_suffixes, "modificateurs à gauche": r_modificateurs, "X-induced": r_induit,
          "sigles définis": r_sigles, "préfixe anti-": r_anti}


def appliquer_regles(pred, documents, regles=tuple(REGLES.values())):
    par_doc = defaultdict(set)
    for m in pred:
        par_doc[m.doc].add(m)
    sortie = set()
    for d in documents:
        P = par_doc[d.document_id]
        for regle in regles:
            P = regle(d, P)
        sortie |= P
    return sortie

In [5]:
pred_v2 = appliquer_regles(dic_v2.extraire_corpus(validation), validation)
attendu = lire_predictions(RESULTATS / "predictions_dictionnaire_validation.jsonl")
attendu = {m for m in attendu if m.doc in textes_val}
assert pred_v2 == attendu, "DIC-v2 n'est pas reproduite"
print("DIC-v2 reproduite à l'identique :", len(pred_v2), "prédictions sur VALIDATION")

DIC-v2 reproduite à l'identique : 7690 prédictions sur VALIDATION


## A — Filtre par taux d'annotation corrigé

Au 2.3, une forme comme « infarction » trouvée à l'intérieur de « myocardial infarction » (annotée) comptait comme
une occurrence non annotée. On ne compte plus ces occurrences incluses dans une mention plus longue.

In [6]:
class DictionnaireCorrige(DictionnaireCDR):
    def construire(self, documents_train):
        super().construire(documents_train)          # entrées et types, comme au 2.3
        types = defaultdict(Counter)
        for d in documents_train:
            for m in d.mentions:
                types[self._cle(m.texte)][m.type] += 1
        self.entrees = {cle: c.most_common(1)[0][0] for cle, c in types.items() if cle}
        self.longueur_max = max(len(k) for k in self.entrees)
        trouvees, annotees = Counter(), Counter()
        for d in documents_train:
            ref = {(m.debut, m.fin, m.type) for m in d.mentions}
            spans = [(m.debut, m.fin) for m in d.mentions]
            for cle, debut, fin, type_ in self._correspondances(d.texte):
                if (debut, fin, type_) not in ref and any(x <= debut and fin <= y and (x, y) != (debut, fin) for x, y in spans):
                    continue                          # occurrence incluse dans une mention plus longue : ignorée
                trouvees[cle] += 1
                annotees[cle] += (debut, fin, type_) in ref
        for cle in self.entrees:
            self.stats[cle]["taux"] = annotees[cle] / trouvees[cle] if trouvees[cle] else 0.0
        self.entrees = {c: t for c, t in self.entrees.items() if self.stats[c]["taux"] >= self.config.seuil_annotation}
        self.longueur_max = max(len(k) for k in self.entrees)
        return self

dic_A = DictionnaireCorrige(CONFIG).construire(train)
print("Entrées : DIC-v2", len(dic_v2.entrees), "→ filtre corrigé", len(dic_A.entrees),
      "| ajoutées :", sorted(" ".join(k) for k in dic_A.entrees.keys() - dic_v2.entrees.keys())[:20])

Entrées : DIC-v2 2348 → filtre corrigé 2375 | ajoutées : ['adenocarcinoma', 'angina', 'arthritis', 'atrophy', 'calcification', 'cancer', 'carcinoma', 'edema', 'fever', 'haemorrhage', 'haemorrhages', 'hemorrhage', 'hemorrhagic', 'hepatitis c', 'ifn', 'infarction', 'infection', 'kidney disease', 'leukemia', 'motor neuropathy']


## B — Variantes singulier / pluriel

Pour chaque entrée, on forme la variante du dernier mot (*tumor* ↔ *tumors*, *anemia* ↔ *anemias*, *-y* ↔ *-ies*).
La variante n'est ajoutée que si elle n'apparaît **jamais** dans les textes de TRAIN : si elle y apparaît sans être
annotée, c'est un indice qu'il ne faut pas l'annoter.

In [7]:
from tals import cle_normalisee

def variante(mot):
    if not mot.isalpha() or len(mot) < 4:
        return None
    if mot.endswith("ies"):
        return mot[:-3] + "y"
    if mot.endswith(("ses", "xes", "ches", "shes")):
        return mot[:-2]
    if mot.endswith("s") and not mot.endswith(("ss", "us", "is")):
        return mot[:-1]
    if mot.endswith("y") and mot[-2] not in "aeiou":
        return mot[:-1] + "ies"
    if mot.endswith(("s", "x", "ch", "sh")):
        return mot + "es"
    return mot + "s"

ngrammes_train = set()
for d in train:
    cles = [cle_normalisee(s) for s, _, _ in tokeniser_avec_offsets(d.texte)]
    for n in range(1, dic_v2.longueur_max + 1):
        ngrammes_train.update(tuple(cles[i:i + n]) for i in range(len(cles) - n + 1))

def ajouter_variantes(dic):
    ajouts = {}
    for cle, type_ in dic.entrees.items():
        v = variante(cle[-1])
        if v:
            cle_v = cle[:-1] + (v,)
            if cle_v not in dic.entrees and cle_v not in ngrammes_train:
                ajouts[cle_v] = type_
    dic.entrees.update(ajouts)
    dic.longueur_max = max(len(k) for k in dic.entrees)
    return ajouts

dic_B = DictionnaireCDR(CONFIG).construire(train)
variantes_B = ajouter_variantes(dic_B)
print(len(variantes_B), "variantes ajoutées, ex. :", sorted(" ".join(k) for k in variantes_B)[:15])

1912 variantes ajoutées, ex. : ['( r ) - alpha - methylhistamines', '1 , 1 - dichloro - 2 , 2 , 2 - trifluoroethanes', '1 , 2 - dimethylhydrazines', '1 , 3 - bis - ( 2 - chloroethyl ) - 1 - nitrosoureas', '1 - ( 2 - methoxyphenyl ) - 4 - [ 4 - ( 2 - phthalimido ) butyl ] piperazines', '1 - [ 2 - ( 3 , 4 - dichlorophenyl ) ethyl ] - 4 - methylpiperazines', '1 - bromo - 1 - chloro - 2 , 2 , 2 - trifluoroethanes', '1 - chloro - 1 , 2 , 2 , 2 - tetrafluoroethanes', '11 - deoxycortisols', '17beta - estradiols', '2 - acetylaminofluorenes', '2 - methoxy - 4 - amino - 5 - chlorobenzoic acid 2 - ( diethylamino ) ethyl esters', '2 - methyl - 6 - ( phenylethynyl ) pyridines', '3 , 4 - dihydroxyphenylacetic acids', '3 , 4 - methylenedioxymethamphetamines']


## C — Extension vers la droite (apprise sur TRAIN)

Symétrique de la règle des modificateurs à gauche : pour chaque correspondance du dictionnaire dans TRAIN, on regarde
les 1 à 3 mots qui suivent. Une suite est retenue si, quand elle suit une forme du même type, elle fait partie de
l'annotation au moins 3 fois et dans au moins 70 % des cas.

In [8]:
def tokens_doc(texte):
    return tokeniser_avec_offsets(texte)

inclus_d, exclus_d = Counter(), Counter()
for d in train:
    toks = tokens_doc(d.texte)
    fin_tok = {b: j for j, (_, _, b) in enumerate(toks)}
    gold = {(m.debut, m.fin, m.type) for m in d.mentions}
    for _, a, b, type_ in dic_v2._correspondances(d.texte):
        j = fin_tok.get(b)
        if j is None:
            continue
        for k in (1, 2, 3):
            if j + k >= len(toks):
                break
            suite = (type_, tuple(cle_normalisee(t[0]) for t in toks[j + 1:j + k + 1]))
            fin = toks[j + k][2]
            if any(x <= a and y == fin and t == type_ for x, y, t in gold):
                inclus_d[suite] += 1
            elif (a, b, type_) in gold:
                exclus_d[suite] += 1
SUITES = {s for s in inclus_d if inclus_d[s] >= 3 and inclus_d[s] / (inclus_d[s] + exclus_d[s]) >= 0.7}
print(len(SUITES), "suites apprises :", sorted((t[:4], " ".join(m)) for t, m in SUITES))

def r_extension_droite(d, P):
    P = set(P)
    toks = tokens_doc(d.texte)
    fin_tok = {b: j for j, (_, _, b) in enumerate(toks)}
    for m in sorted(P, key=lambda m: m.debut):
        j = fin_tok.get(m.fin)
        if j is None or m not in P:
            continue
        for k in (3, 2, 1):
            if j + k < len(toks) and (m.type, tuple(cle_normalisee(t[0]) for t in toks[j + 1:j + k + 1])) in SUITES:
                etendue = Mention(m.doc, m.debut, toks[j + k][2], m.type)
                if libre(etendue, P - {m}):
                    P = (P - {m}) | {etendue}
                break
    return P

2 suites apprises : [('Chem', 'methyl ester'), ('Dise', 'c')]


## D — Propagation dans le document

Une forme retenue dans un résumé (après toutes les règles) est reconnue à ses autres occurrences du même résumé,
avec la même casse et le même type, si la place est libre.

In [9]:
def r_propagation(d, P):
    P = set(P)
    for forme, type_ in {(d.texte[m.debut:m.fin], m.type) for m in P}:
        for r in re.finditer(r"(?<![\w-])" + re.escape(forme) + r"(?![\w-])", d.texte):
            m = Mention(d.document_id, r.start(), r.end(), type_)
            if libre(m, P):
                P.add(m)
    return P

## Effet de chaque ajout sur VALIDATION

Chaque ajout est appliqué seul à DIC-v2, puis tous ensemble. Ordre des règles : suffixes, modificateurs à gauche,
**extension à droite**, « X-induced », sigles, préfixe anti-, **propagation**.

In [10]:
def methode(dic, extension=False, propagation=False):
    global DICO_COURANT
    DICO_COURANT = dic
    regles = [r_suffixes, r_modificateurs] + ([r_extension_droite] if extension else []) + [r_induit, r_sigles, r_anti] \
             + ([r_propagation] if propagation else [])
    return lambda docs: appliquer_regles(dic.extraire_corpus(docs), docs, regles)

def ligne(nom, pred):
    df = tableau_scores(ref_val, pred, "dictionnaire", "validation").set_index("categorie")
    return {"version": nom, "P": df.loc["global (micro)", "precision"], "R": df.loc["global (micro)", "rappel"],
            "F1": df.loc["global (micro)", "F1"], "F1 Chemical": df.loc["Chemical", "F1"], "F1 Disease": df.loc["Disease", "F1"],
            "ajouts justes": len((pred - pred_v2) & ref_val), "ajouts faux": len((pred - pred_v2) - ref_val),
            "pertes": len((pred_v2 - pred) & ref_val)}

dic_AB = DictionnaireCorrige(CONFIG).construire(train)
ajouter_variantes(dic_AB)
VARIANTES = {
    "DIC-v2": methode(dic_v2),
    "+ A filtre corrigé": methode(dic_A),
    "+ B variantes": methode(dic_B),
    "+ C extension à droite": methode(dic_v2, extension=True),
    "+ D propagation": methode(dic_v2, propagation=True),
    "A + B + C + D": methode(dic_AB, extension=True, propagation=True),
}
preds = {nom: f(validation) for nom, f in VARIANTES.items()}
effet = pd.DataFrame([ligne(nom, p) for nom, p in preds.items()])
effet.to_csv(RESULTATS / "dictionnaire_v3_effet_validation.csv", index=False)
effet

,version,P,R,F1,F1 Chemical,F1 Disease,ajouts justes,ajouts faux,pertes
0,DIC-v2,0.8889,0.7128,0.7912,0.8289,0.7434,0,0,0
1,+ A filtre corrigé,0.8841,0.7214,0.7945,0.8293,0.7513,83,57,0
2,+ B variantes,0.8883,0.7180,0.7941,0.8298,0.7493,51,16,1
3,+ C extension à droite,0.8892,0.7130,0.7914,0.8289,0.7440,2,2,0
4,+ D propagation,0.8866,0.7174,0.7931,0.8321,0.7434,45,26,0
5,A + B + C + D,0.8811,0.7314,0.7993,0.8333,0.7569,180,101,1


In [11]:
# Ce que chaque ajout corrige et ce qu'il introduit
def s(m):
    return textes_val[m.doc][m.debut:m.fin]

for nom, p in preds.items():
    if nom == "DIC-v2":
        continue
    print(f"--- {nom}")
    print("  ajouts justes :", Counter(s(m) for m in (p - pred_v2) & ref_val).most_common(8))
    print("  ajouts faux   :", Counter(s(m) for m in (p - pred_v2) - ref_val).most_common(8))
    print("  pertes        :", Counter(s(m) for m in (pred_v2 - p) & ref_val).most_common(5))

--- + A filtre corrigé
  ajouts justes : [('cancer', 14), ('tumors', 12), ('edema', 11), ('hemorrhage', 8), ('infection', 6), ('atrophy', 5), ('fever', 4), ('muscle atrophy', 3)]
  ajouts faux   : [('infection', 11), ('hemorrhagic', 9), ('cancer', 6), ('edema', 5), ('tumors', 5), ('angina', 4), ('fever', 3), ('atrophy', 2)]
  pertes        : []
--- + B variantes
  ajouts justes : [('movement disorders', 6), ('visual disturbances', 5), ('nucleosides', 4), ('CIP', 3), ('pleural effusion', 3), ('diabetics', 2), ('VSDs', 2), ('amphetamines', 2)]
  ajouts faux   : [('tumours', 3), ('epilepsies', 2), ('BP reductions', 1), ('penile erections', 1), ('haematological toxicities', 1), ('myopathies', 1), ('overdoses', 1), ('fit', 1)]
  pertes        : [('toxicities', 1)]
--- + C extension à droite
  ajouts justes : [('hepatitis C', 2)]
  ajouts faux   : [('hepatitis C', 2)]
  pertes        : []
--- + D propagation
  ajouts justes : [('dilevalol', 7), ('ATRA', 5), ('tiapride', 5), ('muscimol', 4), 

## Choix de DIC-v3

On garde les ajouts qui, seuls, améliorent le F1, puis on vérifie que leur combinaison fait au moins aussi bien.

In [12]:
f1 = effet.set_index("version")["F1"]
gardes = [a for a in "ABCD" if f1[[n for n in f1.index if n.startswith(f"+ {a}")][0]] > f1["DIC-v2"]]
print("Ajouts qui améliorent le F1 seuls :", gardes)

dic_v3 = DictionnaireCorrige(CONFIG).construire(train) if "A" in gardes else DictionnaireCDR(CONFIG).construire(train)
VARIANTES_AJOUTEES = ajouter_variantes(dic_v3) if "B" in gardes else {}
methode_v3 = methode(dic_v3, extension="C" in gardes, propagation="D" in gardes)

t0 = time.perf_counter()
pred_val_v3 = methode_v3(validation)
t_val = time.perf_counter() - t0
scores_v3 = tableau_scores(ref_val, pred_val_v3, "dictionnaire_v3", "validation", t_val)
scores_v3.to_csv(RESULTATS / "scores_dictionnaire_v3_validation.csv", index=False)
ecrire_predictions(pred_val_v3, "dictionnaire_v3", validation, RESULTATS / "predictions_dictionnaire_v3_validation.jsonl")
display(pd.concat([tableau_scores(ref_val, pred_v2, "DIC-v2", "validation"), scores_v3], ignore_index=True))
pd.DataFrame({n: {k: len(v) for k, v in erreurs(ref_val, p).items()} for n, p in [("DIC-v2", pred_v2), ("DIC-v3", pred_val_v3)]}).T

Ajouts qui améliorent le F1 seuls : ['A', 'B', 'C', 'D']


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,DIC-v2,validation,Chemical,3999,303,1348,0.9296,0.7479,0.8289,False,None
1,DIC-v2,validation,Disease,2837,551,1407,0.8374,0.6685,0.7434,False,None
2,DIC-v2,validation,global (micro),6836,854,2755,0.8889,0.7128,0.7912,False,None
3,dictionnaire_v3,validation,Chemical,4055,330,1292,0.9247,0.7584,0.8333,False,NaN
4,dictionnaire_v3,validation,Disease,2960,617,1284,0.8275,0.6975,0.7569,False,NaN
5,dictionnaire_v3,validation,global (micro),7015,947,2576,0.8811,0.7314,0.7993,False,4.03


,correcte,frontiere,type,manquee,parasite
DIC-v2,6836,576,8,2171,270
DIC-v3,7015,627,8,1941,312


## Gel et prédictions sur TEST

Dictionnaire, variantes et règles sont construits sur TRAIN, puis appliqués aux textes de TEST ; le score TEST sera
calculé dans `5_evaluation.ipynb`.

In [13]:
t0 = time.perf_counter()
pred_test_v3 = methode_v3(test)
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test_v3, "dictionnaire_v3", test, RESULTATS / "predictions_dictionnaire_v3_test.jsonl")

gel = {"version": "DIC-v3", "base": "DIC-v2 (notebook 2.3), reproduite à l'identique", "config": vars(CONFIG),
       "ajouts_gardes": gardes, "entrees": len(dic_v3.entrees), "variantes_ajoutees": len(VARIANTES_AJOUTEES),
       "suites_droite": sorted([t, " ".join(m)] for t, m in SUITES) if "C" in gardes else [],
       "temps_s": {"extraction_validation": round(t_val, 2), "extraction_test": round(t_test, 2)},
       "predictions_test": len(pred_test_v3)}
(RESULTATS / "dictionnaire_v3_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in gel.items() if k != "suites_droite"}, ensure_ascii=False, indent=2))

{
  "version": "DIC-v3",
  "base": "DIC-v2 (notebook 2.3), reproduite à l'identique",
  "config": {
    "casse": "insensible",
    "seuil_annotation": 0.3,
    "regex": true
  },
  "ajouts_gardes": [
    "A",
    "B",
    "C",
    "D"
  ],
  "entrees": 4303,
  "variantes_ajoutees": 1928,
  "temps_s": {
    "extraction_validation": 4.03,
    "extraction_test": 3.32
  },
  "predictions_test": 8290
}
